In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import plotly.express as px

pd.set_option("display.max_columns", None)

In [2]:
customers = pd.read_csv("../data/processed/customers_clean.csv")
orders = pd.read_csv("../data/processed/orders_clean.csv", parse_dates=[
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
])

order_items = pd.read_csv("../data/processed/order_items_clean.csv")
payments = pd.read_csv("../data/processed/payments_clean.csv")
products = pd.read_csv("../data/processed/products_clean.csv")
categories = pd.read_csv("../data/processed/categories_clean.csv")

In [3]:
master = (
    orders
    .merge(customers,on="customer_id")
    .merge(order_items,on="order_id")
    .merge(products,on="product_id",how="left")
    .merge(payments,on="order_id",how="left")
)

In [4]:
master.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,payment_sequential,payment_type,payment_installments,payment_value
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,1,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0,1.0,credit_card,1.0,18.12
1,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,1,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0,3.0,voucher,1.0,2.00
2,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,1,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0,2.0,voucher,1.0,18.59
3,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,1,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76,perfumaria,29.0,178.0,1.0,400.0,19.0,13.0,19.0,1.0,boleto,1.0,141.46
4,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,1,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22,automotivo,46.0,232.0,1.0,420.0,24.0,19.0,21.0,1.0,credit_card,3.0,179.12


In [5]:
master.shape

(117604, 30)

In [6]:
master["payment_value"].sum()

np.float64(20308134.71)

In [7]:
master["order_id"].nunique()

98666

In [8]:
master["customer_unique_id"].nunique()

95420

In [9]:
master.groupby("order_id")["payment_value"].sum().mean()

np.float64(205.82708035189427)

In [10]:
monthly_sales = (
    master
    .groupby(master["order_purchase_timestamp"].dt.to_period("M"))
    ["payment_value"]
    .sum()
    .reset_index()
)

monthly_sales["order_purchase_timestamp"] = monthly_sales["order_purchase_timestamp"].astype(str)

In [11]:
px.line(
    monthly_sales,
    x="order_purchase_timestamp",
    y="payment_value",
    title="Monthly Revenue"
)

In [12]:
monthly_orders = (
    master
    .groupby(master["order_purchase_timestamp"].dt.to_period("M"))
    ["order_id"]
    .nunique()
)

In [15]:
monthly_orders = (
    master
    .groupby(master["order_purchase_timestamp"].dt.to_period("M"))
    ["order_id"]
    .nunique()
    .reset_index(name="order_count")
)

monthly_orders["order_purchase_timestamp"] = (
    monthly_orders["order_purchase_timestamp"].astype(str)
)

In [16]:
fig = px.line(
    monthly_orders,
    x="order_purchase_timestamp",
    y="order_count",
    title="Monthly Orders",
    labels={
        "order_purchase_timestamp": "Month",
        "order_count": "Number of Orders"
    },
    markers=True
)

fig.show()

In [17]:
order_status = (
    master["order_status"]
    .value_counts()
    .reset_index()
)

order_status.columns = ["order_status", "count"]

In [18]:
fig = px.pie(
    order_status,
    names="order_status",
    values="count",
    title="Order Status Distribution"
)

fig.show()

In [19]:
top_states = (
    master.groupby("customer_state")["payment_value"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

In [20]:
fig = px.bar(
    top_states,
    x="customer_state",
    y="payment_value",
    title="Top 10 States by Revenue",
    labels={
        "customer_state":"State",
        "payment_value":"Revenue"
    }
)

fig.show()

In [21]:
top_cities = (
    master.groupby("customer_city")["payment_value"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

In [22]:
fig = px.bar(
    top_cities,
    x="customer_city",
    y="payment_value",
    title="Top 10 Cities by Revenue"
)

fig.show()

In [23]:
payment = (
    master["payment_type"]
    .value_counts()
    .reset_index()
)

payment.columns = ["payment_type","count"]

In [24]:
fig = px.pie(
    payment,
    names="payment_type",
    values="count",
    title="Payment Method Distribution"
)

fig.show()

In [25]:
master = master.merge(
    categories,
    on="product_category_name",
    how="left"
)

In [26]:
top_categories = (
    master["product_category_name_english"]
    .value_counts()
    .head(10)
    .reset_index()
)

top_categories.columns = ["category","orders"]

In [27]:
fig = px.bar(
    top_categories,
    x="orders",
    y="category",
    orientation="h",
    title="Top Product Categories"
)

fig.show()

In [28]:
category_revenue = (
    master.groupby("product_category_name_english")["payment_value"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

In [29]:
fig = px.bar(
    category_revenue,
    x="payment_value",
    y="product_category_name_english",
    orientation="h",
    title="Top Categories by Revenue"
)

fig.show()